# CS-781 Capstone — Phase 2: Legal Named Entity Recognition (L-NER)

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to find the 12 legal entity types and writes one `submission.zip` per experiment.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`** (your Hugging Face token) for this notebook.
3. **Add Input:** attach the output of each of your four Phase-1 notebooks (step-by-step instructions are in the chat). Section 6 prints which attached file is used for A, B, C and D.
4. *Save Version → Save & Run All.*

## Your four Phase-1 models and the four submissions
| Model | Phase-1 notebook | Design |
|---|---|---|
| A | `phase1-a` | GPT decoder, first run (220M tokens) |
| B | `phase1-b` | GQA + RoPE + RMSNorm + SwiGLU decoder |
| C | `phase1-c` | encoder-decoder (its encoder is used) |
| D | `phase1-d-optimisedA` | GPT decoder, optimised run (400M tokens, better data and decoding settings) |

| Submission | Model | Fine-tuning technique | Head |
|---|---|---|---|
| S1_D_lora | D | LoRA | BiLSTM + CRF |
| S2_B_lorma | B | **LoRMA** (compulsory) | BiLSTM + CRF |
| S3_C_adapter | C | Adapters | BiLSTM + CRF |
| S4_A4_kd | A (first 4 layers) | **Knowledge distillation**, teacher S1 | BiLSTM + CRF |

## What it does
| Section | Step |
|---|---|
| 3 | Loads the Phase-2 data and checks the offset convention on the gold annotations |
| 4–5 | Splits text into **pieces** with exact character offsets, converts entities to BIO tags, and reports how many gold entities can be reproduced exactly (the ceiling of this approach) |
| 6 | Rebuilds the four Phase-1 architectures and loads their trained weights |
| 7 | Fine-tuning techniques: full, partial, **LoRA**, **LoRMA**, adapters, BitFit |
| 8 | NER heads: BiLSTM + **CRF**, softmax token classifier, start/end **pointer** (span extraction) |
| 9–10 | Trains each experiment: dev stage (train fold 1, pick the best epoch on fold 2), then retrains on both folds and predicts the test set |
| 11–12 | Comparison table (score, per-label F1, error types, parameters, time, memory) and the submission files |

## Rules this follows (course document §8)
Only your Phase-1 models are used as starting points; no other pretrained model, weights or API. Training uses only the provided L-NER folds. The four submissions use four different techniques, including the compulsory **LoRMA** and **knowledge distillation**. (The document asks for six experiments; extra experiments are defined in Section 2 if you need them.)

## Re-running
Everything finished is loaded instead of redone: tokenisation, each training stage (resumes from the last epoch), test predictions. To continue in a new session, attach this notebook's previous output with *Add Input*. Change `RUN_EXPERIMENTS` in Section 2 to choose what runs. `CS781_P2_SMOKE=1` runs a tiny test of the whole pipeline.

## 1. Setup and Hugging Face login

In [1]:
import os
import re
import sys
import json
import math
import time
import glob
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
PyTorch: 2.11.0+cu128 | device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration and experiments

Each experiment is one line: which Phase-1 model, which fine-tuning technique, which head. `RUN_EXPERIMENTS` lists what runs, in order (the four submissions S1–S4 by default). `PHASE1_MODELS` says which attached Phase-1 file is your A, B, C and D.

In [2]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE2_DIR = "Phase_2_LNER"

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P2_WORK_ROOT",
                           "/kaggle/working/cs781_phase2" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase2"))
LOCAL_DATA_DIR = os.environ.get("CS781_P2_DATA_DIR", "")
os.makedirs(WORK_ROOT, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}

DEFAULTS = dict(
    backbone="D",
    n_layers=None,
    technique="full",
    context="bilstm",
    decoder="crf",
    max_tokens=1024,
    window_overlap=0.5,
    batch_size=8,
    epochs=15, patience=5,
    lr_backbone=5e-5,
    lr_peft=5e-4,
    lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    dropout=0.1,
    lstm_hidden=256, lstm_layers=1,
    partial_top_layers=4,
    lora_r=16, lora_alpha=32,
    peft_targets="qv",
    lorma_variant="plus",
    adapter_dim=64,
    teacher=None,
    kd_temperature=2.0, kd_weight=0.5,
    class_weighting=False,
    pointer_pos_weight=2.0,
    seed=SEED,
)

EXPERIMENTS = {
    "S1_D_lora":             dict(backbone="D", technique="lora"),
    "S2_B_lorma":            dict(backbone="B", technique="lorma"),
    "S3_C_adapter":          dict(backbone="C", technique="adapter"),
    "S4_A4_kd":              dict(backbone="A", n_layers=4, technique="full", lstm_hidden=128,
                                  teacher="S1_D_lora"),
    "E1_D_full":             dict(backbone="D", technique="full"),
    "E4_D_partial_pointer":  dict(backbone="D", technique="partial", decoder="pointer"),
    "X1_D_full_softmax":     dict(backbone="D", technique="full", context="none", decoder="softmax"),
    "X2_B_bitfit":           dict(backbone="B", technique="bitfit"),
    "X3_A4_no_teacher":      dict(backbone="A", n_layers=4, technique="full", lstm_hidden=128),
    "X4_D_lorma":            dict(backbone="D", technique="lorma"),
    "X5_D_lorma_pi":         dict(backbone="D", technique="lorma", lorma_variant="pi"),
    "X6_A_lora":             dict(backbone="A", technique="lora"),
}
RUN_EXPERIMENTS = ["S1_D_lora", "S2_B_lorma", "S3_C_adapter", "S4_A4_kd"]
if os.environ.get("CS781_P2_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P2_RUN"].split(",") if x]

FINAL_MODE = "retrain_all"
DEV_TRAIN_FOLD, DEV_EVAL_FOLD = "fold1", "fold2"

SMOKE_TEST = os.environ.get("CS781_P2_SMOKE", "0") == "1"
if SMOKE_TEST:
    DEFAULTS.update(epochs=3, patience=3, max_tokens=96, batch_size=4, lstm_hidden=32, lora_r=4, adapter_dim=8,
                    partial_top_layers=1, lr_backbone=1e-3, lr_peft=3e-3, lr_head=3e-3)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        _cfg.pop("lstm_hidden", None)
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    return cfg

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase2":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-2 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    _c = experiment_config(_name)
    print(f"  {_name:22s} model {_c['backbone']}{'' if not _c['n_layers'] else ' (first ' + str(_c['n_layers']) + ' layers)'} | "
          f"{_c['technique']:8s} | {_c['context']} + {_c['decoder']}" + (f" | teacher {_c['teacher']}" if _c['teacher'] else ""))

Working folder: /kaggle/working/cs781_phase2
Experiments to run: ['S1_D_lora', 'S2_B_lorma', 'S3_C_adapter', 'S4_A4_kd']
  S1_D_lora              model D | lora     | bilstm + crf
  S2_B_lorma             model B | lorma    | bilstm + crf
  S3_C_adapter           model C | adapter  | bilstm + crf
  S4_A4_kd               model A (first 4 layers) | full     | bilstm + crf | teacher S1_D_lora


## 3. Load the L-NER data

Gold annotations are `[start, end, label, text]`. The cell checks on the data whether `end` is exclusive (Python slicing) or inclusive, and uses the same convention for the predictions.

In [3]:
def phase2_file(relative):
    """Local path of a Phase-2 file (attached dataset, local folder, or Hugging Face download)."""
    candidates = []
    if LOCAL_DATA_DIR:
        candidates.append(os.path.join(LOCAL_DATA_DIR, PHASE2_DIR, relative))
        candidates.append(os.path.join(LOCAL_DATA_DIR, relative))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(1, 5):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, *(["*"] * depth), PHASE2_DIR, relative)))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{PHASE2_DIR}/{relative}", repo_type="dataset", token=HF_TOKEN)

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def as_text(value):
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        for key in ("text", "document", "content"):
            if isinstance(value.get(key), str):
                return value[key]
    raise TypeError("Unexpected document format: " + str(type(value)))

def parse_entity(item):
    """One gold annotation -> (start, end, label, entity_text or None)."""
    if isinstance(item, dict):
        start = item.get("start", item.get("start_character", item.get("start_char")))
        end = item.get("end", item.get("end_character", item.get("end_char")))
        label = item.get("label", item.get("entity_label", item.get("type")))
        text = item.get("text", item.get("entity_text"))
    else:
        start, end, label = item[0], item[1], item[2]
        text = item[3] if len(item) > 3 else None
    return int(start), int(end), str(label), (text if isinstance(text, str) else None)

TEXTS, GOLD_RAW = {}, {}
for fold in ("fold1", "fold2"):
    TEXTS[fold] = {str(k): as_text(v) for k, v in load_json_file(phase2_file(f"Train/{fold}-text.json")).items()}
    GOLD_RAW[fold] = {str(k): [parse_entity(x) for x in v] for k, v in load_json_file(phase2_file(f"Train/{fold}-gold.json")).items()}
TEXTS["fold3"] = {str(k): as_text(v) for k, v in load_json_file(phase2_file("Test/fold3-text.json")).items()}

_exclusive = _inclusive = _with_text = 0
for fold in GOLD_RAW:
    for doc_id, entities in GOLD_RAW[fold].items():
        text = TEXTS[fold].get(doc_id, "")
        for start, end, _, ent_text in entities:
            if ent_text is not None:
                _with_text += 1
                _exclusive += text[start:end] == ent_text
                _inclusive += text[start:end + 1] == ent_text
END_INCLUSIVE = _with_text > 0 and _inclusive > _exclusive
print(f"Offset check on {_with_text:,} annotations with text: end-exclusive matches {_exclusive:,} | end-inclusive matches {_inclusive:,}"
      f" -> end offset is {'INCLUSIVE' if END_INCLUSIVE else 'EXCLUSIVE'}")
if _with_text and max(_exclusive, _inclusive) < 0.98 * _with_text:
    print("WARNING: many annotation texts do not match their offsets; look at the data before trusting the scores.")

GOLD = {fold: {doc_id: [(s, e + 1 if END_INCLUSIVE else e, lab) for s, e, lab, _ in ents] for doc_id, ents in docs.items()}
        for fold, docs in GOLD_RAW.items()}
for fold in ("fold1", "fold2"):
    for doc_id in TEXTS[fold]:
        GOLD[fold].setdefault(doc_id, [])

_seen = Counter(lab for fold in GOLD for ents in GOLD[fold].values() for _, _, lab in ents)
try:
    with open(phase2_file("Train/labels.txt"), "r", encoding="utf-8") as f:
        _raw = [line.strip() for line in f if line.strip()]
    LABELS = []
    for lab in _raw:
        lab = re.sub(r"^[BIES]-", "", lab)
        if lab and lab != "O" and lab not in LABELS:
            LABELS.append(lab)
except Exception as e:
    print("labels.txt not readable (" + type(e).__name__ + "); using the labels found in the annotations.")
    LABELS = sorted(_seen)
for lab in sorted(_seen):
    if lab not in LABELS:
        print(f"NOTE: label {lab!r} appears in the annotations but not in labels.txt; it is added.")
        LABELS.append(lab)
LABEL_TO_ID = {lab: i for i, lab in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

for fold in ("fold1", "fold2", "fold3"):
    n_chars = [len(t) for t in TEXTS[fold].values()]
    n_ents = sum(len(v) for v in GOLD.get(fold, {}).values())
    print(f"{fold}: {len(TEXTS[fold])} documents | characters per document: mean {np.mean(n_chars):,.0f}, max {max(n_chars):,}"
          + (f" | {n_ents:,} entities" if fold in GOLD else " | test (no labels)"))
print(f"{NUM_LABELS} labels:", LABELS)
print("Entities per label:", dict(sorted(_seen.items(), key=lambda kv: -kv[1])))

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Offset check on 8,393 annotations with text: end-exclusive matches 8,393 | end-inclusive matches 17 -> end offset is EXCLUSIVE


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1: 35 documents | characters per document: mean 31,360, max 106,947 | 4,120 entities
fold2: 35 documents | characters per document: mean 31,887, max 114,044 | 4,273 entities
fold3: 35 documents | characters per document: mean 33,509, max 97,103 | test (no labels)
12 labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Entities per label: {'DATE': 1575, 'STAT': 1260, 'PREC': 1183, 'COURT': 1013, 'AUTH': 1003, 'CASENO': 760, 'APP': 407, 'RESP': 381, 'JUDGE': 245, 'A.COUNSEL': 195, 'WIT': 187, 'R.COUNSEL': 184}


## 4. Tokenizer and pieces

The Phase-1 BPE tokenizer is loaded, not retrained. Text is cut into **pieces** (a run of letters, a run of digits, or one other character) so that entity boundaries next to punctuation can be predicted exactly.

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

PIECE_RE = re.compile(r"[^\W\d_]+|\d+|[^\w\s]|_")

def split_pieces(text):
    return [(m.start(), m.end()) for m in PIECE_RE.finditer(text)]

def encode_document(text, tokenizer):
    """-> dict with the piece character spans, the token ids, and for every piece its token positions."""
    spans = split_pieces(text)
    n = len(spans)
    tokens = []
    tok_start = np.zeros(n, dtype=np.int64)
    tok_last = np.zeros(n, dtype=np.int64)
    tok_stop = np.zeros(n, dtype=np.int64)
    for i, (s, e) in enumerate(spans):
        tok_start[i] = len(tokens)
        tokens.extend(tokenizer.encode_piece(text[s:e]))
        tok_last[i] = len(tokens) - 1
        if i == n - 1 or spans[i + 1][0] > e:
            tokens.append(tokenizer.eow_id)
        tok_stop[i] = len(tokens)
    return {"spans": np.asarray(spans, dtype=np.int64).reshape(-1, 2), "tokens": np.asarray(tokens, dtype=np.int64),
            "tok_start": tok_start, "tok_last": tok_last, "tok_stop": tok_stop}

_probe = "Mr. A.K. Sen, Sr. Adv.(for the appellant) in C.A. No.123/2010 dated 12.03.2015 - Rs.5,000/-"
_spans = split_pieces(_probe)
assert "".join(_probe[s:e] for s, e in _spans) == "".join(_probe.split()), "piece splitting lost characters"
print("Piece example:", [_probe[s:e] for s, e in _spans][:16], "...")

Piece example: ['Mr', '.', 'A', '.', 'K', '.', 'Sen', ',', 'Sr', '.', 'Adv', '.', '(', 'for', 'the', 'appellant'] ...


## 5. Tags, alignment check and the metric

Entities become BIO tags over pieces. The check prints the best score this scheme could reach if the model were perfect. The scorer is the official rule: an entity counts only if start, end and label all match; F1 is averaged over the 12 labels.

In [5]:
NUM_TAGS = 1 + 2 * NUM_LABELS
ALL_DOCS = {(fold, doc_id): text for fold in TEXTS for doc_id, text in TEXTS[fold].items()}
PIECES = {key: np.asarray(split_pieces(text), dtype=np.int64).reshape(-1, 2) for key, text in ALL_DOCS.items()}

def entities_to_tags(spans, entities, stats=None):
    """Character-level entities -> one BIO tag per piece. Returns (tags, list of (piece_start, piece_end, label_id))."""
    tags = np.zeros(len(spans), dtype=np.int64)
    piece_entities = []
    if len(spans) == 0:
        return tags, piece_entities
    starts, ends = spans[:, 0], spans[:, 1]
    taken = np.zeros(len(spans), dtype=bool)
    for s, e, lab in sorted(entities, key=lambda x: (x[0], -(x[1] - x[0]))):
        first = int(np.searchsorted(ends, s, side="right"))
        last = int(np.searchsorted(starts, e, side="left")) - 1
        if stats is not None:
            stats["entities"] += 1
        if first > last or lab not in LABEL_TO_ID:
            if stats is not None:
                stats["dropped"] += 1
            continue
        aligned = starts[first] == s and ends[last] == e
        if taken[first:last + 1].any():
            if stats is not None:
                stats["overlapping"] += 1
            continue
        if stats is not None and not aligned:
            stats["misaligned"] += 1
            if len(stats["examples"]) < 5:
                stats["examples"].append((s, e, lab))
        k = LABEL_TO_ID[lab]
        tags[first] = 1 + 2 * k
        tags[first + 1:last + 1] = 2 + 2 * k
        taken[first:last + 1] = True
        piece_entities.append((first, last + 1, k))
    return tags, piece_entities

def tags_to_piece_entities(tags):
    out, start, label = [], None, None
    for i, t in enumerate(list(tags) + [0]):
        t = int(t)
        is_b, k = (t % 2 == 1), (t - 1) // 2
        if start is not None and (t == 0 or is_b or k != label):
            out.append((start, i, label))
            start = None
        if t != 0 and (is_b or start is None):
            start, label = i, k
    return out

def piece_entities_to_chars(spans, piece_entities):
    return [(int(spans[a, 0]), int(spans[b - 1, 1]), LABELS[k]) for a, b, k in piece_entities]

def strict_scores(gold, pred):
    """gold / pred: {doc_key: [(start, end, label), ...]} -> per-label precision / recall / F1 and their macro average."""
    tp, n_pred, n_gold = Counter(), Counter(), Counter()
    for key in set(gold) | set(pred):
        g, p = set(gold.get(key, [])), set(pred.get(key, []))
        for _, _, lab in g:
            n_gold[lab] += 1
        for _, _, lab in p:
            n_pred[lab] += 1
        for _, _, lab in g & p:
            tp[lab] += 1
    per_label = {}
    for lab in LABELS:
        precision = tp[lab] / n_pred[lab] if n_pred[lab] else 0.0
        recall = tp[lab] / n_gold[lab] if n_gold[lab] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[lab] = {"precision": precision, "recall": recall, "f1": f1,
                          "tp": tp[lab], "predicted": n_pred[lab], "gold": n_gold[lab]}
    total_tp, total_pred, total_gold = sum(tp.values()), sum(n_pred.values()), sum(n_gold.values())
    micro_p = total_tp / total_pred if total_pred else 0.0
    micro_r = total_tp / total_gold if total_gold else 0.0
    return {"macro_f1": float(np.mean([v["f1"] for v in per_label.values()])),
            "macro_precision": float(np.mean([v["precision"] for v in per_label.values()])),
            "macro_recall": float(np.mean([v["recall"] for v in per_label.values()])),
            "micro_f1": 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0,
            "per_label": per_label}

def error_analysis(gold, pred):
    """Why predictions are wrong: boundary errors, label confusions, spurious and missed entities."""
    counts, confusion = Counter(), Counter()
    for key in set(gold) | set(pred):
        g, p = gold.get(key, []), pred.get(key, [])
        gset, pset = set(g), set(p)
        gspan = {(s, e): lab for s, e, lab in g}
        for s, e, lab in p:
            if (s, e, lab) in gset:
                counts["correct"] += 1
            elif (s, e) in gspan:
                counts["wrong_label_exact_span"] += 1
                confusion[(gspan[(s, e)], lab)] += 1
            else:
                overlap = [(gs, ge, gl) for gs, ge, gl in g if gs < e and s < ge]
                if any(gl == lab for _, _, gl in overlap):
                    counts["boundary_error_right_label"] += 1
                elif overlap:
                    counts["boundary_and_label_error"] += 1
                else:
                    counts["spurious"] += 1
        for s, e, lab in g:
            if (s, e, lab) not in pset and not any(ps < e and s < pe for ps, pe, _ in p):
                counts["missed_completely"] += 1
    return {"counts": dict(counts), "top_label_confusions": [[a, b, n] for (a, b), n in confusion.most_common(8)]}

GOLD_TAGS, GOLD_PIECE_ENTITIES = {}, {}
_stats = {"entities": 0, "misaligned": 0, "overlapping": 0, "dropped": 0, "examples": []}
for fold in GOLD:
    for doc_id, entities in GOLD[fold].items():
        key = (fold, doc_id)
        GOLD_TAGS[key], GOLD_PIECE_ENTITIES[key] = entities_to_tags(PIECES[key], entities, _stats)
_round_trip = {key: piece_entities_to_chars(PIECES[key], tags_to_piece_entities(GOLD_TAGS[key])) for key in GOLD_TAGS}
_gold_all = {(fold, d): ents for fold in GOLD for d, ents in GOLD[fold].items()}
ORACLE = strict_scores(_gold_all, _round_trip)
MAX_ENTITY_PIECES = max([b - a for ents in GOLD_PIECE_ENTITIES.values() for a, b, _ in ents] + [1])
print(f"Gold entities: {_stats['entities']:,} | not on a piece boundary: {_stats['misaligned']:,} | overlapping another entity: "
      f"{_stats['overlapping']:,} | empty / unknown label: {_stats['dropped']:,}")
print(f"Best score this tagging scheme can reach on the training folds (gold -> tags -> spans): "
      f"macro-F1 {ORACLE['macro_f1']:.4f}, micro-F1 {ORACLE['micro_f1']:.4f}")
print("Longest entity:", MAX_ENTITY_PIECES, "pieces | pieces per document: mean",
      int(np.mean([len(v) for v in PIECES.values()])), "max", max(len(v) for v in PIECES.values()))
if _stats["examples"]:
    for s, e, lab in _stats["examples"]:
        print(f"   misaligned example [{lab}] offsets {s}-{e}")
if ORACLE["micro_f1"] < 0.98:
    print("WARNING: more than 2% of gold entities cannot be expressed exactly; inspect the misaligned examples above.")

Gold entities: 8,393 | not on a piece boundary: 61 | overlapping another entity: 0 | empty / unknown label: 0
Best score this tagging scheme can reach on the training folds (gold -> tags -> spans): macro-F1 0.9944, micro-F1 0.9927
Longest entity: 42 pieces | pieces per document: mean 6561 max 23264
   misaligned example [AUTH] offsets 1963-1976
   misaligned example [AUTH] offsets 11218-11231
   misaligned example [STAT] offsets 17853-17865
   misaligned example [AUTH] offsets 25083-25099
   misaligned example [AUTH] offsets 25450-25466


## 6. Phase-1 models

Finds the attached Phase-1 checkpoints, prints which one is your A, B, C and D, and rebuilds the four architectures with identical parameter names, so the trained weights load exactly. The models return the hidden states of all layers instead of next-token logits.

In [6]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        states = []
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

Phase-1 checkpoints found under /kaggle/input: 4
  GPT decoder              val loss 1.3086 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-D/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GPT decoder              val loss 1.7840 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GQA/RoPE decoder         val loss 1.6521 | 264 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-B/cs781_phase1/B_GQA_RoPE_RMSNorm_SwiGLU/legal-model-final
  encoder-decoder          val loss 3.0858 | 422 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-C/cs781_phase1/C_EncoderDecoder/legal-model-final
Your Phase-1 models:
  A: GPT decoder, validation loss 1.7840 | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  B: GQA/RoPE decoder, validation loss 1.6521

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `full` | every weight |
| `partial` | the top *k* layers |
| `lora` | low-rank matrices **added** to the query/value weights: W = W0 + (α/r)·BA |
| `lorma` | low-rank matrices that **multiply** the query/value weights: W = (I + (α/r)·BA)·W0 (LoRMA+) |
| `adapter` | small bottleneck layers after the attention and feed-forward sub-layers (Houlsby) |
| `bitfit` | bias terms and normalisation gains only |

The NER head is always trained.

In [7]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero."""

    def __init__(self, base, r, alpha, rows=None):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                parts.append(hs + self.scale * F.linear(F.linear(hs, A.to(hs.dtype)), B.to(hs.dtype)))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_")

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows)
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "adapter":
        for block in blocks:
            for path in SUBLAYER_OUTPUTS[arch]:
                _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. NER heads

A CRF over BIO tags, a plain softmax token classifier, or start/end pointers (span extraction).

In [8]:
def bio_allowed():
    allowed = torch.ones(NUM_TAGS, NUM_TAGS, dtype=torch.bool)
    for k in range(NUM_LABELS):
        inside = 2 + 2 * k
        allowed[:, inside] = False
        allowed[1 + 2 * k, inside] = True
        allowed[inside, inside] = True
    return allowed

NEG = -1e4
INSIDE_TAGS = [2 + 2 * k for k in range(NUM_LABELS)]

class CRF(nn.Module):
    """Linear-chain conditional random field over the BIO tags. It scores whole tag sequences, so it learns which
    tag transitions are likely, and impossible ones (O -> I-X, B-X -> I-Y) are forbidden."""

    def __init__(self):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(NUM_TAGS, NUM_TAGS))
        self.start = nn.Parameter(torch.zeros(NUM_TAGS))
        self.end = nn.Parameter(torch.zeros(NUM_TAGS))
        self.register_buffer("allowed", bio_allowed(), persistent=False)

    def constrained(self):
        return self.transitions.masked_fill(~self.allowed, NEG)

    def nll(self, emissions, tags, mask):
        """Negative log-likelihood summed over the batch. emissions [B, P, K], tags [B, P], mask [B, P] (bool)."""
        trans = self.constrained()
        batch = torch.arange(emissions.size(0), device=emissions.device)
        fmask = mask.float()
        score = self.start[tags[:, 0]] + emissions[batch, 0, tags[:, 0]]
        alpha = self.start[None] + emissions[:, 0]
        for t in range(1, emissions.size(1)):
            score = score + (emissions[batch, t, tags[:, t]] + trans[tags[:, t - 1], tags[:, t]]) * fmask[:, t]
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + emissions[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        last = tags[batch, mask.long().sum(1) - 1]
        score = score + self.end[last]
        log_z = torch.logsumexp(alpha + self.end[None], dim=1)
        return (log_z - score).sum()

def viterbi(emissions, transitions, start, end):
    """Best tag sequence for one document. emissions [P, K] (numpy), transitions [K, K], start [K], end [K]."""
    n = emissions.shape[0]
    score = start + emissions[0]
    back = np.zeros((n, emissions.shape[1]), dtype=np.int32)
    for t in range(1, n):
        cand = score[:, None] + transitions
        back[t] = cand.argmax(0)
        score = cand.max(0) + emissions[t]
    tag = int((score + end).argmax())
    path = [tag]
    for t in range(n - 1, 0, -1):
        tag = int(back[t, tag])
        path.append(tag)
    return path[::-1]

def decode_pointer(logits, max_len, threshold=0.5):
    """Span extraction: logits [P, 2 * labels] = start scores then end scores. Each predicted start is paired with the
    nearest predicted end of the same label; overlapping spans are resolved by confidence."""
    probs = 1.0 / (1.0 + np.exp(-logits.astype(np.float64)))
    p_start, p_end = probs[:, :NUM_LABELS], probs[:, NUM_LABELS:]
    candidates = []
    for k in range(NUM_LABELS):
        starts, ends = np.flatnonzero(p_start[:, k] > threshold), np.flatnonzero(p_end[:, k] > threshold)
        for s in starts:
            j = np.searchsorted(ends, s)
            if j < len(ends) and ends[j] - s < max_len:
                candidates.append((p_start[s, k] * p_end[ends[j], k], int(s), int(ends[j]) + 1, k))
    taken = np.zeros(len(logits), dtype=bool)
    out = []
    for _, s, e, k in sorted(candidates, reverse=True):
        if not taken[s:e].any():
            taken[s:e] = True
            out.append((s, e, k))
    return sorted(out)

class NERModel(nn.Module):
    """Phase-1 language model + NER head.

    1. The language model turns the tokens into hidden states. A learned weighted average over ALL its layers is
       used ("scalar mix"), because the last layer of a language model is specialised for next-word prediction.
    2. Each piece is represented by the hidden state of its last token (causal models: that token has seen the
       whole piece) or the mean of its first and last token (bidirectional encoder).
    3. Optional BiLSTM over the pieces. Models A, B and D read left-to-right only, so a piece cannot see the words
       after it; the BiLSTM adds that right-hand context ("... , counsel for the appellant").
    4. A linear layer gives a score per tag, decoded with a CRF, a softmax, or start/end pointers.
    """

    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone, self.decoder_type = backbone, cfg["decoder"]
        d = backbone.d_model
        self.mix = nn.Parameter(torch.zeros(len(backbone.layers())))
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.lstm = None
        if cfg["context"] == "bilstm":
            self.lstm = nn.LSTM(d, cfg["lstm_hidden"], num_layers=cfg["lstm_layers"], batch_first=True, bidirectional=True)
            d = 2 * cfg["lstm_hidden"]
        self.n_out = 2 * NUM_LABELS if self.decoder_type == "pointer" else NUM_TAGS
        self.out = nn.Linear(d, self.n_out)
        self.crf = CRF() if self.decoder_type == "crf" else None

    def forward(self, batch):
        """-> scores [B, P, n_out] in float32."""
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = self.backbone(batch["input_ids"], batch["pad_mask"])
        weights = torch.softmax(self.mix, dim=0)
        h = sum(w * s.float() for w, s in zip(weights, states))
        index = batch["piece_last"][:, :, None].expand(-1, -1, h.size(-1))
        pieces = h.gather(1, index)
        if self.backbone.bidirectional:
            first = batch["piece_first"][:, :, None].expand(-1, -1, h.size(-1))
            pieces = 0.5 * (pieces + h.gather(1, first))
        pieces = self.drop(self.norm(pieces))
        if self.lstm is not None:
            lengths = batch["piece_mask"].sum(1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(pieces, lengths, batch_first=True, enforce_sorted=False)
            pieces, _ = nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True,
                                                         total_length=batch["piece_mask"].size(1))
            pieces = self.drop(pieces)
        return self.out(pieces)

    def loss(self, scores, batch, cfg, class_weights=None):
        mask = batch["piece_mask"]
        n = mask.sum().clamp(min=1)
        if self.decoder_type == "crf":
            hard = self.crf.nll(scores, batch["tags"], mask) / n
        elif self.decoder_type == "softmax":
            hard = F.cross_entropy(scores[mask], batch["tags"][mask], weight=class_weights)
        else:
            pos_weight = torch.full((self.n_out,), float(cfg["pointer_pos_weight"]), device=scores.device)
            hard = F.binary_cross_entropy_with_logits(scores[mask], batch["pointer_targets"][mask], pos_weight=pos_weight)
        if "teacher" not in batch:
            return hard, hard.detach(), None
        T = float(cfg["kd_temperature"])
        s, t = scores[mask] / T, batch["teacher"][mask] / T
        if self.decoder_type == "pointer":
            soft = F.binary_cross_entropy_with_logits(s, torch.sigmoid(t)) * T * T
        else:
            soft = F.kl_div(F.log_softmax(s, dim=-1), F.softmax(t, dim=-1), reduction="batchmean") * T * T
        w = float(cfg["kd_weight"])
        return (1 - w) * hard + w * soft, hard.detach(), soft.detach()

    def decode(self, scores):
        """Document-level scores [P, n_out] (numpy) -> list of (piece_start, piece_end, label_id)."""
        if len(scores) == 0:
            return []
        if self.decoder_type == "pointer":
            return decode_pointer(scores, max_len=int(1.5 * MAX_ENTITY_PIECES) + 2)
        start = np.zeros(NUM_TAGS, dtype=np.float64)
        end = np.zeros(NUM_TAGS, dtype=np.float64)
        if self.decoder_type == "crf":
            transitions = self.crf.constrained().detach().float().cpu().numpy().astype(np.float64)
            start += self.crf.start.detach().float().cpu().numpy()
            end += self.crf.end.detach().float().cpu().numpy()
            emissions = scores.astype(np.float64)
        else:
            transitions = np.where(bio_allowed().numpy(), 0.0, NEG)
            shifted = scores.astype(np.float64) - scores.max(axis=1, keepdims=True)
            emissions = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        start[INSIDE_TAGS] = NEG
        return tags_to_piece_entities(viterbi(emissions, transitions, start, end))

## 9. Windows, training and inference

Long judgments are processed in overlapping windows. For every piece, the prediction comes from the window where the piece is furthest from an edge. Decoding runs once over the whole document, so entities are never cut at window borders.

In [9]:
import gc

ENCODINGS = {}
POINTER_TARGETS = {}
BIG = 10 ** 9

def encodings_for(slot, keys):
    tokenizer = tokenizer_for(slot)
    cache = ENCODINGS.setdefault(PHASE1[slot]["tokenizer_dir"], {})
    missing = [k for k in keys if k not in cache]
    for key in tqdm(missing, desc=f"tokenizing for model {slot}", disable=len(missing) < 5):
        cache[key] = encode_document(ALL_DOCS[key], tokenizer)
    return cache

def build_windows(enc, max_tokens, overlap):
    """Split one document into overlapping windows of pieces [(first_piece, last_piece + 1), ...]."""
    n, out, s = len(enc["tok_stop"]), [], 0
    while n > 0:
        e = int(np.searchsorted(enc["tok_stop"], enc["tok_start"][s] + max_tokens, side="right"))
        e = max(e, s + 1)
        out.append((s, e))
        if e >= n:
            break
        s += max(1, int(round((e - s) * (1.0 - overlap))))
    return out

def pointer_targets(key):
    if key not in POINTER_TARGETS:
        target = np.zeros((len(PIECES[key]), 2 * NUM_LABELS), dtype=np.float32)
        for a, b, k in GOLD_PIECE_ENTITIES.get(key, []):
            target[a, k] = 1.0
            target[b - 1, NUM_LABELS + k] = 1.0
        POINTER_TARGETS[key] = target
    return POINTER_TARGETS[key]

def collate(items, encodings, max_tokens, with_labels, decoder_type, teacher_scores=None):
    """items: [(doc_key, first_piece, last_piece + 1)] -> padded tensors on the GPU."""
    rows = []
    for key, s, e in items:
        enc = encodings[key]
        t0 = int(enc["tok_start"][s])
        t1 = min(int(enc["tok_stop"][e - 1]), t0 + max_tokens)
        rows.append((key, s, e, enc["tokens"][t0:t1],
                     np.minimum(enc["tok_start"][s:e] - t0, t1 - t0 - 1), np.minimum(enc["tok_last"][s:e] - t0, t1 - t0 - 1)))
    B, T, P = len(rows), max(len(r[3]) for r in rows), max(r[2] - r[1] for r in rows)
    input_ids = np.full((B, T), PAD_ID, dtype=np.int64)
    pad_mask = np.zeros((B, T), dtype=bool)
    piece_first, piece_last = np.zeros((B, P), dtype=np.int64), np.zeros((B, P), dtype=np.int64)
    piece_mask = np.zeros((B, P), dtype=bool)
    tags = np.zeros((B, P), dtype=np.int64)
    extra = np.zeros((B, P, 2 * NUM_LABELS), dtype=np.float32) if (with_labels and decoder_type == "pointer") else None
    teacher = None if teacher_scores is None else np.zeros((B, P, next(iter(teacher_scores.values())).shape[1]), dtype=np.float32)
    for i, (key, s, e, ids, first, last) in enumerate(rows):
        n = e - s
        input_ids[i, :len(ids)], pad_mask[i, :len(ids)] = ids, True
        piece_first[i, :n], piece_last[i, :n], piece_mask[i, :n] = first, last, True
        if with_labels:
            tags[i, :n] = GOLD_TAGS[key][s:e]
            if extra is not None:
                extra[i, :n] = pointer_targets(key)[s:e]
        if teacher is not None:
            teacher[i, :n] = teacher_scores[key][s:e]
    batch = {"input_ids": torch.from_numpy(input_ids).to(DEVICE), "pad_mask": torch.from_numpy(pad_mask).to(DEVICE),
             "piece_first": torch.from_numpy(piece_first).to(DEVICE), "piece_last": torch.from_numpy(piece_last).to(DEVICE),
             "piece_mask": torch.from_numpy(piece_mask).to(DEVICE)}
    if with_labels:
        batch["tags"] = torch.from_numpy(tags).to(DEVICE)
        if extra is not None:
            batch["pointer_targets"] = torch.from_numpy(extra).to(DEVICE)
    if teacher is not None:
        batch["teacher"] = torch.from_numpy(teacher).to(DEVICE)
    return batch

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    backbone = load_backbone(cfg["backbone"], cfg["n_layers"], cfg["dropout"], verbose=verbose)
    apply_technique(backbone, PHASE1[cfg["backbone"]]["arch"], cfg)
    return NERModel(backbone, cfg).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone = sum(p.numel() for p in model.backbone.parameters())
    backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / total,
            "language_model": backbone, "language_model_trainable": backbone_trainable, "head": total - backbone}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k != "name"}, "phase1": [entry["config"], entry["eval_loss"]],
               "labels": LABELS, "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    backbone_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg["lr_head"]
        elif any(marker in name for marker in NEW_WEIGHT_MARKERS):
            lr = cfg["lr_peft"]
        else:
            lr = backbone_lr
        decay = cfg["weight_decay"] if (p.ndim >= 2 and "crf." not in name) else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

@torch.no_grad()
def predict_scores(model, cfg, keys, encodings, desc="predict"):
    """Run the model over every window and keep, for each piece, the scores from the window in which the piece is
    furthest from a window edge (it has the most context there). -> {doc_key: float32 [pieces, n_out]}"""
    model.eval()
    max_tokens = min(cfg["max_tokens"], model.backbone.max_tokens)
    items = [(key, s, e) for key in keys for s, e in build_windows(encodings[key], max_tokens, cfg["window_overlap"])]
    items.sort(key=lambda it: int(encodings[it[0]]["tok_stop"][it[2] - 1] - encodings[it[0]]["tok_start"][it[1]]))
    scores = {key: np.zeros((len(PIECES[key]), model.n_out), dtype=np.float32) for key in keys}
    best = {key: np.full(len(PIECES[key]), -1, dtype=np.int64) for key in keys}
    batch_size = 2 * cfg["batch_size"]
    for i in tqdm(range(0, len(items), batch_size), desc=desc, leave=False):
        chunk = items[i:i + batch_size]
        out = model(collate(chunk, encodings, max_tokens, False, model.decoder_type)).float().cpu().numpy()
        for row, (key, s, e) in zip(out, chunk):
            n, total = e - s, len(PIECES[key])
            position = np.arange(n)
            left = position if s > 0 else np.full(n, BIG, dtype=np.int64)
            right = (n - 1 - position) if e < total else np.full(n, BIG, dtype=np.int64)
            centrality = np.minimum(left, right)
            better = centrality > best[key][s:e]
            scores[key][s:e][better] = row[:n][better]
            best[key][s:e][better] = centrality[better]
    return scores

def scores_to_entities(model, scores):
    return {key: piece_entities_to_chars(PIECES[key], model.decode(doc_scores)) for key, doc_scores in scores.items()}

def evaluate(model, cfg, keys, encodings, desc="evaluate"):
    scores = predict_scores(model, cfg, keys, encodings, desc)
    predicted = scores_to_entities(model, scores)
    gold = {key: GOLD[key[0]][key[1]] for key in keys}
    return strict_scores(gold, predicted), predicted, gold

def train_stage(cfg, stage, train_keys, eval_keys, n_epochs, signature, teacher_scores=None):
    """Train (or resume, or load) one model. stage = "dev" (train fold 1, choose the best epoch on fold 2) or
    "final" (train on both folds for a fixed number of epochs). Returns (model, info)."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, f"{stage}_last.pt"), os.path.join(folder, f"{stage}_model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    params = count_parameters(model)
    print(f"[{cfg['name']} / {stage}] parameters: {params['total'] / 1e6:.2f}M total | trained: "
          f"{params['language_model_trainable'] / 1e6:.3f}M of the {params['language_model'] / 1e6:.2f}M language-model weights "
          f"+ {params['head'] / 1e6:.2f}M NER head ({params['trainable_percent']:.2f}% of all weights)")
    encodings = encodings_for(cfg["backbone"], list(train_keys) + list(eval_keys))
    max_tokens = min(cfg["max_tokens"], model.backbone.max_tokens)
    windows = [(key, s, e) for key in train_keys for s, e in build_windows(encodings[key], max_tokens, cfg["window_overlap"])]
    steps_per_epoch = math.ceil(len(windows) / cfg["batch_size"])
    total_steps = steps_per_epoch * cfg["epochs"]
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    class_weights = None
    if cfg["class_weighting"] and cfg["decoder"] == "softmax":
        counts = np.bincount(np.concatenate([GOLD_TAGS[k] for k in train_keys]), minlength=NUM_TAGS).astype(np.float64)
        weights = 1.0 / np.sqrt(counts + 100.0)
        class_weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=DEVICE)

    state = {"epoch": 0, "history": [], "best_f1": -1.0, "best_epoch": 0, "best_metrics": None, "bad_epochs": 0, "seconds": 0.0,
             "peak_memory_mb": 0.0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']} / {stage}] resuming after epoch {state['epoch']}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    stopped_early = False
    for epoch in range(state["epoch"], n_epochs):
        if eval_keys and state["bad_epochs"] >= cfg["patience"]:
            stopped_early = True
            break
        t0 = time.time()
        model.train()
        order = list(windows)
        random.Random(cfg["seed"] * 1000 + epoch).shuffle(order)
        torch.manual_seed(cfg["seed"] * 1000 + epoch)
        loss_sum, hard_sum, soft_sum, n_batches = 0.0, 0.0, 0.0, 0
        bar = tqdm(range(0, len(order), cfg["batch_size"]), desc=f"{cfg['name']} {stage} epoch {epoch + 1}/{n_epochs}", leave=False)
        for b, i in enumerate(bar):
            step = epoch * steps_per_epoch + b
            factor = lr_factor(step, total_steps, cfg["warmup_frac"])
            for group in optimizer.param_groups:
                group["lr"] = group["base_lr"] * factor
            batch = collate(order[i:i + cfg["batch_size"]], encodings, max_tokens, True, cfg["decoder"], teacher_scores)
            loss, hard, soft = model.loss(model(batch), batch, cfg, class_weights)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item()
            hard_sum += hard.item()
            soft_sum += soft.item() if soft is not None else 0.0
            n_batches += 1
            if b % 10 == 0:
                bar.set_postfix(loss=f"{loss_sum / n_batches:.4f}")
        record = {"epoch": epoch + 1, "train_loss": loss_sum / max(1, n_batches), "label_loss": hard_sum / max(1, n_batches)}
        if teacher_scores is not None:
            record["distillation_loss"] = soft_sum / max(1, n_batches)
        state["epoch"] = epoch + 1
        message = f"[{cfg['name']} / {stage}] epoch {epoch + 1:2d}/{n_epochs} | train loss {record['train_loss']:.4f}"
        if eval_keys:
            metrics, _, _ = evaluate(model, cfg, eval_keys, encodings, desc="validation")
            record.update(dev_macro_f1=metrics["macro_f1"], dev_macro_precision=metrics["macro_precision"],
                          dev_macro_recall=metrics["macro_recall"], dev_micro_f1=metrics["micro_f1"])
            message += (f" | dev strict macro-F1 {metrics['macro_f1']:.4f} (P {metrics['macro_precision']:.4f}, "
                        f"R {metrics['macro_recall']:.4f}) | micro-F1 {metrics['micro_f1']:.4f}")
            if metrics["macro_f1"] > state["best_f1"]:
                state.update(best_f1=metrics["macro_f1"], best_epoch=epoch + 1, best_metrics=metrics, bad_epochs=0)
                save_atomic({"model": trainable_state(model), "signature": signature, "epoch": epoch + 1}, model_path)
                message += "  <- best"
            else:
                state["bad_epochs"] += 1
        message += f" | {time.time() - t0:.0f} s"
        state["history"].append(record)
        state["seconds"] += time.time() - t0
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message, flush=True)

    if eval_keys:
        if stopped_early or state["bad_epochs"] >= cfg["patience"]:
            print(f"[{cfg['name']} / {stage}] early stop: no improvement for {cfg['patience']} epochs.")
        load_trainable(model, torch_load(model_path)["model"])
    else:
        save_atomic({"model": trainable_state(model), "signature": signature, "epoch": state["epoch"]}, model_path)
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_epoch": state["best_epoch"] or state["epoch"], "history": state["history"], "metrics": state["best_metrics"],
            "train_seconds": state["seconds"], "peak_memory_mb": state["peak_memory_mb"], "parameters": params,
            "train_windows": len(windows)}
    return model, info

def load_stage_model(cfg, stage):
    model = build_model(cfg, verbose=False)
    load_trainable(model, torch_load(os.path.join(exp_dir(cfg["name"]), f"{stage}_model.pt"))["model"])
    return model

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 10. Run the experiments

For each experiment: dev stage → final stage → test predictions → `submission.zip`. With a `teacher`, the student also learns from the teacher's softened tag scores (knowledge distillation).

In [10]:
FOLD_KEYS = {fold: [(fold, doc_id) for doc_id in TEXTS[fold]] for fold in TEXTS}
RESULTS = {}

def teacher_scores_for(cfg, stage, keys):
    """Scores of the teacher model for the student's training documents (knowledge distillation). In the dev stage
    the teacher trained on fold 1 is used, so fold 2 stays unseen; in the final stage the teacher trained on both."""
    teacher_cfg = experiment_config(cfg["teacher"])
    if teacher_cfg["decoder"] == "pointer" or cfg["decoder"] == "pointer":
        assert teacher_cfg["decoder"] == cfg["decoder"], "teacher and student need the same kind of decoder"
    teacher_stage = stage if os.path.exists(os.path.join(exp_dir(teacher_cfg["name"]), f"{stage}_model.pt")) else "dev"
    teacher = load_stage_model(teacher_cfg, teacher_stage)
    scores = predict_scores(teacher, teacher_cfg, keys, encodings_for(teacher_cfg["backbone"], keys), desc="teacher scores")
    print(f"[{cfg['name']} / {stage}] distilling from {teacher_cfg['name']} ({teacher_stage} model, "
          f"{count_parameters(teacher)['total'] / 1e6:.1f}M parameters)")
    release(teacher)
    return scores

def write_predictions(folder, predicted):
    """prediction.json: {doc_id: [[start, end, label], ...]} for every test document, zipped as submission.zip."""
    output = {}
    for fold, doc_id in FOLD_KEYS["fold3"]:
        spans = sorted(predicted.get((fold, doc_id), []))
        output[doc_id] = [[int(s), int(e - 1 if END_INCLUSIVE else e), lab] for s, e, lab in spans]
    prediction_path, zip_path = os.path.join(folder, "prediction.json"), os.path.join(folder, "submission.zip")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        check = json.loads(z.read("prediction.json").decode("utf-8"))
        assert z.namelist() == ["prediction.json"]
    assert set(check) == set(TEXTS["fold3"]), "a test document is missing"
    for doc_id, spans in check.items():
        length = len(TEXTS["fold3"][doc_id])
        for s, e, lab in spans:
            assert lab in LABEL_TO_ID and 0 <= s <= e <= length, (doc_id, s, e, lab)
    return zip_path, sum(len(v) for v in output.values())

def run_experiment(name):
    """Dev stage -> final stage -> test predictions. Every finished part is loaded from disk instead of redone."""
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    teacher_signature = ""
    if cfg["teacher"]:
        teacher_result = run_experiment(cfg["teacher"])
        if teacher_result is None:
            print(f"[{name}] SKIPPED: its teacher {cfg['teacher']} could not be run.")
            return None
        teacher_signature = teacher_result["signature"]
    folder = exp_dir(name)
    dev_signature = config_signature(cfg, "dev" + teacher_signature)
    signature = config_signature(cfg, FINAL_MODE + teacher_signature)
    results_path, dev_path = os.path.join(folder, "results.json"), os.path.join(folder, "dev_results.json")
    if os.path.exists(results_path) and os.path.exists(os.path.join(folder, "submission.zip")):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature:
            print(f"[{name}] already finished: dev strict macro-F1 {saved['dev']['macro_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    train_keys, dev_keys = FOLD_KEYS[DEV_TRAIN_FOLD], FOLD_KEYS[DEV_EVAL_FOLD]
    model = None
    dev = load_json_file(dev_path) if os.path.exists(dev_path) else None
    if dev is None or dev.get("signature") != dev_signature or not os.path.exists(os.path.join(folder, "dev_model.pt")):
        teacher = teacher_scores_for(cfg, "dev", train_keys) if cfg["teacher"] else None
        model, info = train_stage(cfg, "dev", train_keys, dev_keys, cfg["epochs"], dev_signature, teacher)
        metrics, predicted, gold = evaluate(model, cfg, dev_keys, encodings_for(cfg["backbone"], dev_keys), "dev evaluation")
        dev = {"signature": dev_signature, "best_epoch": info["best_epoch"], "history": info["history"],
               "macro_f1": metrics["macro_f1"], "macro_precision": metrics["macro_precision"],
               "macro_recall": metrics["macro_recall"], "micro_f1": metrics["micro_f1"], "per_label": metrics["per_label"],
               "errors": error_analysis(gold, predicted), "parameters": info["parameters"],
               "train_seconds": info["train_seconds"], "peak_memory_mb": info["peak_memory_mb"],
               "train_windows": info["train_windows"]}
        write_json(dev, dev_path)
        write_json({doc_id: [[s, e, lab] for s, e, lab in spans] for (_, doc_id), spans in predicted.items()},
                   os.path.join(folder, "dev_predictions.json"))
    else:
        print(f"[{name}] dev stage already done: best epoch {dev['best_epoch']}, strict macro-F1 {dev['macro_f1']:.4f}")
    print(f"[{name}] DEV strict macro-F1 {dev['macro_f1']:.4f} | precision {dev['macro_precision']:.4f} | "
          f"recall {dev['macro_recall']:.4f} | best epoch {dev['best_epoch']}")

    final_seconds = 0.0
    if FINAL_MODE == "retrain_all":
        release(model)
        all_keys = train_keys + dev_keys
        final_path = os.path.join(folder, "final_model.pt")
        if os.path.exists(final_path) and torch_load(final_path).get("signature") == signature:
            print(f"[{name}] final model already trained; loading it.")
            model = load_stage_model(cfg, "final")
        else:
            teacher = teacher_scores_for(cfg, "final", all_keys) if cfg["teacher"] else None
            model, info = train_stage(cfg, "final", all_keys, [], dev["best_epoch"], signature, teacher)
            final_seconds = info["train_seconds"]
    elif model is None:
        model = load_stage_model(cfg, "dev")

    test_keys = FOLD_KEYS["fold3"]
    encodings = encodings_for(cfg["backbone"], test_keys)
    t0 = time.time()
    predicted = scores_to_entities(model, predict_scores(model, cfg, test_keys, encodings, "test prediction"))
    inference_seconds = time.time() - t0
    zip_path, n_entities = write_predictions(folder, predicted)
    n_pieces = sum(len(PIECES[k]) for k in test_keys)
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "final_mode": FINAL_MODE, "dev": dev,
              "final_train_seconds": final_seconds, "test_entities": n_entities,
              "inference_seconds": inference_seconds, "inference_pieces_per_second": n_pieces / max(1e-9, inference_seconds),
              "submission": zip_path}
    write_json(result, results_path)
    print(f"[{name}] test predictions: {n_entities:,} entities in {len(test_keys)} documents -> {zip_path}\n")
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(experiment_config(_name)["seed"])
    run_experiment(_name)

Loaded Phase-1 model D (GPT decoder): 8/8 layers, 64.57M parameters
[S1_D_lora / dev] parameters: 67.08M total | trained: 0.393M of the 64.96M language-model weights + 2.12M NER head (3.74% of all weights)


tokenizing for model D:   0%|          | 0/70 [00:00<?, ?it/s]

S1_D_lora dev epoch 1/15:   0%|          | 0/109 [00:00<?, ?it/s]

validation:   0%|          | 0/55 [00:00<?, ?it/s]

[S1_D_lora / dev] epoch  1/15 | train loss 0.6184 | dev strict macro-F1 0.1812 (P 0.2501, R 0.1592) | micro-F1 0.3829  <- best | 94 s


S1_D_lora dev epoch 2/15:   0%|          | 0/109 [00:00<?, ?it/s]

... (228 outputs left out)

[S4_A4_kd / final] epoch 12/12 | train loss 0.0098 | 134 s


tokenizing for model A:   0%|          | 0/35 [00:00<?, ?it/s]

test prediction:   0%|          | 0/58 [00:00<?, ?it/s]

[S4_A4_kd] test predictions: 3,515 entities in 35 documents -> /kaggle/working/cs781_phase2/S4_A4_kd/submission.zip



## 11. Results and analysis

In [11]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["dev"]["macro_f1"]):
    cfg, dev = r["config"], r["dev"]
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": cfg["technique"] + (" + KD" if cfg["teacher"] else ""), "head": cfg["context"] + "+" + cfg["decoder"],
                 "dev_macro_f1": dev["macro_f1"], "dev_macro_p": dev["macro_precision"], "dev_macro_r": dev["macro_recall"],
                 "dev_micro_f1": dev["micro_f1"], "best_epoch": dev["best_epoch"],
                 "params_M": dev["parameters"]["total"] / 1e6, "trainable_M": dev["parameters"]["trainable"] / 1e6,
                 "lm_trained_M": dev["parameters"].get("language_model_trainable", 0) / 1e6,
                 "trainable_%": dev["parameters"]["trainable_percent"], "train_min": dev["train_seconds"] / 60,
                 "peak_mem_GB": dev["peak_memory_mb"] / 1000, "pieces_per_s": r["inference_pieces_per_second"]})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':24s} {'model':9s} {'technique':14s} {'head':16s} {'macroF1':>8s} {'P':>7s} {'R':>7s} {'microF1':>8s} "
              f"{'params':>8s} {'trainable':>10s} {'LM part':>9s} {'%':>6s} {'train':>7s} {'mem':>6s} {'speed':>9s}")
    print("Dev fold (fold 2), strict span + label match. 'trainable' = all trained weights (language model + NER head); "
          "'LM part' = trained weights inside the language model.\n" + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:24s} {x['model']:9s} {x['technique']:14s} {x['head']:16s} {x['dev_macro_f1']:8.4f} "
              f"{x['dev_macro_p']:7.4f} {x['dev_macro_r']:7.4f} {x['dev_micro_f1']:8.4f} {x['params_M']:7.1f}M "
              f"{x['trainable_M']:9.3f}M {x['lm_trained_M']:8.3f}M {x['trainable_%']:6.2f} {x['train_min']:6.1f}m {x['peak_mem_GB']:5.1f}G "
              f"{x['pieces_per_s']:8,.0f}/s")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(f"{v:.6g}" if isinstance(v, float) else str(v) for v in x.values()) + "\n")

    print("\nF1 per label (dev fold):")
    names = [x["experiment"] for x in rows]
    print(f"{'label':12s} {'gold':>6s} " + " ".join(f"{n[:12]:>12s}" for n in names))
    for lab in LABELS:
        gold_count = finished[names[0]]["dev"]["per_label"][lab]["gold"]
        print(f"{lab:12s} {gold_count:6d} " + " ".join(f"{finished[n]['dev']['per_label'][lab]['f1']:12.4f}" for n in names))

    print("\nError types on the dev fold (counts of predicted entities; 'missed' counts gold entities with no overlap):")
    kinds = ["correct", "boundary_error_right_label", "wrong_label_exact_span", "boundary_and_label_error", "spurious", "missed_completely"]
    print(f"{'experiment':24s} " + " ".join(f"{k[:14]:>14s}" for k in kinds))
    for n in names:
        counts = finished[n]["dev"]["errors"]["counts"]
        print(f"{n:24s} " + " ".join(f"{counts.get(k, 0):14d}" for k in kinds))
    best = names[0]
    print(f"\nMost frequent label confusions of {best} (gold -> predicted, same span):",
          finished[best]["dev"]["errors"]["top_label_confusions"])
    print("\nSummary table saved to", os.path.join(WORK_ROOT, "results_summary.csv"))

Dev fold (fold 2), strict span + label match. 'trainable' = all trained weights (language model + NER head); 'LM part' = trained weights inside the language model.
experiment               model     technique      head              macroF1       P       R  microF1   params  trainable   LM part      %   train    mem     speed
------------------------------------------------------------------------------------------------------------------------------------------------------------------
S2_B_lorma               B         lorma          bilstm+crf         0.4903  0.5389  0.4733   0.6414    68.5M     2.378M    0.262M   3.47   24.5m   2.9G   13,565/s
S1_D_lora                D         lora           bilstm+crf         0.4793  0.5395  0.4604   0.6388    67.1M     2.510M    0.393M   3.74   22.2m   2.4G   15,143/s
S4_A4_kd                 A[:4]     full + KD      bilstm+crf         0.4753  0.5444  0.4477   0.6346    37.1M    37.146M   36.217M 100.00   19.0m   1.9G   27,855/s
S3_C_adapter      

## 12. Submission files

In [12]:
if rows:
    print("Submission files:")
    for x in rows:
        print(f"  {x['experiment']:24s} dev macro-F1 {x['dev_macro_f1']:.4f}  {finished[x['experiment']]['submission']}")
    best_zip = os.path.join(WORK_ROOT, rows[0]["experiment"], "submission.zip")
    if os.path.exists(best_zip):
        shutil.copy2(best_zip, os.path.join(WORK_ROOT, "submission.zip"))
        print(f"\nBest on dev: {rows[0]['experiment']} -> copied to {os.path.join(WORK_ROOT, 'submission.zip')}")

Submission files:
  S2_B_lorma               dev macro-F1 0.4903  /kaggle/working/cs781_phase2/S2_B_lorma/submission.zip
  S1_D_lora                dev macro-F1 0.4793  /kaggle/working/cs781_phase2/S1_D_lora/submission.zip
  S4_A4_kd                 dev macro-F1 0.4753  /kaggle/working/cs781_phase2/S4_A4_kd/submission.zip
  S3_C_adapter             dev macro-F1 0.0000  /kaggle/working/cs781_phase2/S3_C_adapter/submission.zip

Best on dev: S2_B_lorma -> copied to /kaggle/working/cs781_phase2/submission.zip
